# Indonesian Spellcheck — Character N-gram Method

Complementary spellchecker to the pyspellchecker baseline. While pyspellchecker
asks "is this exact word in the vocabulary?", this method asks "is this word's
character pattern consistent with Indonesian phonotactics?"

The two methods catch different kinds of errors:
- **pyspellchecker** flags words missing from the dictionary (good for OCR
  errors that produce unique garbage)
- **n-gram method** flags words with unusual character patterns (good for
  OCR errors that produce plausibly-spelled but unattested words, and for
  flagging rare-but-real words that pyspellchecker accepts)

## Method

1. **Train** a character trigram model on the words in the chosen Indonesian
   dictionary (`indonesian_dict_strategy_B.json` or `..._C.json`). Use
   add-k smoothing (k=0.01) to handle unseen trigrams.
2. **Score** every Indonesian token in every Parcor file. The score is the
   word's **perplexity** — a standard NLP metric where higher = more
   suspicious. Lower-bounded by a baseline (perfect predictability) and
   upper-bounded by infinity (impossible patterns).
3. **Threshold** at the 95th percentile of training-vocabulary perplexity.
   Words above this threshold are flagged. The threshold is data-derived,
   not hand-picked.
4. **Output** per-token scores, per-file aggregates, and corpus-level top-N
   most suspicious tokens.

## Strategy switch

Same pattern as pyspellchecker v2: choose one source dict by uncommenting
its `DICT_PATH` line. Run multiple times to compare strategies.

## Output

For each Parcor file, produces a `*_Parcor_ngram.csv` with columns:
- `kalimat_asal`, `kalimat_tujuan` — original sentences
- `indonesian_side` — `asal` or `tujuan`
- `total_tokens`
- `flagged_tokens` — semicolon-separated tokens above the perplexity threshold
- `n_flagged` — count
- `flag_rate` — fraction
- `perplexities` — semicolon-separated `token:perplexity` pairs for flagged words
- `mean_perplexity` — average perplexity across all tokens in the row

Plus per-strategy `_aggregate.csv` and `_top_suspicious_tokens.csv`.

## How this complements pyspellchecker

After running both methods on the same files, you can ask:
- Which words are flagged by **both** methods? (high-confidence errors)
- Which by **only pyspellchecker**? (genuinely-unknown but well-formed words)
- Which by **only n-gram**? (in vocab but oddly-shaped — possibly noise that
  the dict-builder admitted)
- Which by **neither**? (clean words)

These four categories are the basis of the comparison story for the thesis.

## 1. Configuration

In [30]:
import os
import re
import json
import math
import time
from pathlib import Path
from collections import Counter, defaultdict
import concurrent.futures

import pandas as pd
import numpy as np

# === Choose ONE strategy by uncommenting ===
# DICT_PATH = "../spellCheckDicts/indonesian_dict.json"
# DICT_PATH = "../spellCheckDicts/indonesian_dict_strategy_B.json"
DICT_PATH = "../spellCheckDicts/indonesian_dict_strategy_C.json"

# === Paths ===
SRC_PARCOR_DIR = Path("../Ekstraksi/11. Parallel Corpus - Fixed")
LOOKUP_PATH    = Path("../Ekstraksi/12. Parallel Corpus - Spelling Checker/LookupIsFromIndonesia.csv")

strategy_tag = Path(DICT_PATH).stem.replace("indonesian_dict", "").lstrip("_") or "original"
DST_DIR = Path(f"../Ekstraksi/12. Parallel Corpus - Spellcheck Ngram/{strategy_tag}")
DST_DIR.mkdir(parents=True, exist_ok=True)

# === N-gram model parameters ===
NGRAM_ORDER = 3
SMOOTHING_K = 0.01
PADDING_CHAR = "^"   # at word boundaries
END_CHAR = "$"

# === Threshold ===
THRESHOLD_PERCENTILE = 95   # flag words whose perplexity > this percentile of training words

# === Performance ===
MAX_WORKERS = 4

print(f"Dictionary:    {DICT_PATH}")
print(f"Strategy tag:  {strategy_tag}")
print(f"N-gram order:  {NGRAM_ORDER} (smoothing k={SMOOTHING_K})")
print(f"Output dir:    {DST_DIR.resolve()}")

assert os.path.exists(DICT_PATH), f"Dictionary not found: {DICT_PATH}"
assert SRC_PARCOR_DIR.exists(), f"Parcor dir not found: {SRC_PARCOR_DIR}"

Dictionary:    ../spellCheckDicts/indonesian_dict_strategy_C.json
Strategy tag:  strategy_C
N-gram order:  3 (smoothing k=0.01)
Output dir:    C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\12. Parallel Corpus - Spellcheck Ngram\strategy_C


## 2. Load training vocabulary

Read the chosen Indonesian dictionary. We use both the words AND their
frequencies — frequent words contribute more weight to the n-gram model
(matching how the dict was built with tier weighting).

In [31]:
with open(DICT_PATH, "r", encoding="utf-8") as f:
    vocab = json.load(f)

# Filter: require length >= 2 and alphabetic (matches dict_builder rules)
vocab = {w: int(f) for w, f in vocab.items() if len(w) >= 2 and w.isalpha()}

print(f"Loaded {len(vocab):,} unique training words")
print(f"Total weighted token count: {sum(vocab.values()):,}")
print(f"Frequency range: {min(vocab.values())}–{max(vocab.values())}")

Loaded 22,601 unique training words
Total weighted token count: 256,712
Frequency range: 3–48


## 3. Build the trigram model

Steps:
1. Pad each word: `abadi` becomes `^^abadi$$` (so we can score the first
   and last trigrams as containing word-boundary information).
2. Count all trigrams across the vocabulary, weighted by word frequency.
3. Apply add-k smoothing: every possible trigram gets `+k` count, even
   unseen ones. This prevents zero-probabilities for unseen test trigrams.
4. Compute log-probabilities for each trigram, given its preceding bigram.

In [32]:
def make_padded(word: str) -> str:
    """Pad word with NGRAM_ORDER-1 boundary chars on each side."""
    pad = PADDING_CHAR * (NGRAM_ORDER - 1)
    end = END_CHAR * (NGRAM_ORDER - 1)
    return pad + word + end


def trigrams(padded: str):
    """Yield each trigram in the padded word."""
    for i in range(len(padded) - NGRAM_ORDER + 1):
        yield padded[i : i + NGRAM_ORDER]


# Count trigrams (and the bigrams they're conditioned on)
trigram_counts = Counter()       # full trigram -> count
context_counts = Counter()       # bigram (first 2 chars of trigram) -> count

for word, freq in vocab.items():
    padded = make_padded(word)
    for t in trigrams(padded):
        trigram_counts[t] += freq
        context_counts[t[:NGRAM_ORDER - 1]] += freq

# Vocabulary of characters (for smoothing denominator)
char_vocab = set()
for word in vocab:
    char_vocab.update(word)
char_vocab.add(PADDING_CHAR)
char_vocab.add(END_CHAR)
V = len(char_vocab)

print(f"Unique trigrams seen:    {len(trigram_counts):,}")
print(f"Unique contexts (bigrams): {len(context_counts):,}")
print(f"Character vocabulary size: {V}")


def trigram_logprob(trigram: str) -> float:
    """
    Returns log P(trigram[2] | trigram[:2]) with add-k smoothing.

    Add-k formula: P(c | ctx) = (count(ctx + c) + k) / (count(ctx) + k * V)
    """
    context = trigram[: NGRAM_ORDER - 1]
    numerator = trigram_counts.get(trigram, 0) + SMOOTHING_K
    denominator = context_counts.get(context, 0) + SMOOTHING_K * V
    return math.log(numerator / denominator)

Unique trigrams seen:    4,177
Unique contexts (bigrams): 513
Character vocabulary size: 28


## 4. Score a word

The word's **perplexity** is the standard metric:

$$\text{perplexity}(w) = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(t_i)\right)$$

where $t_i$ are the trigrams in the padded word. Length-normalized, so it's
comparable across words of different lengths.

- Perplexity ≈ 1: very predictable (word looks exactly like training data)
- Perplexity moderate: typical Indonesian word
- Perplexity very high: unusual character patterns (likely OCR error)

We also report `avg_neg_logprob` (the simpler interpretation: just the mean
of -log(P) across trigrams). It's the perplexity in log space.

In [33]:
def score_word(word: str) -> dict:
    """
    Returns {
        'perplexity': float,
        'avg_neg_logprob': float,
        'n_trigrams': int,
    }
    """
    padded = make_padded(word.lower())
    log_probs = [trigram_logprob(t) for t in trigrams(padded)]
    if not log_probs:
        return {"perplexity": float("inf"), "avg_neg_logprob": float("inf"), "n_trigrams": 0}
    avg_log = sum(log_probs) / len(log_probs)
    return {
        "perplexity": math.exp(-avg_log),
        "avg_neg_logprob": -avg_log,
        "n_trigrams": len(log_probs),
    }


# Sanity check
for w in ["mengabaikan", "rumah", "yang", "ekstghak", "petigh", "tughun", "xqz"]:
    s = score_word(w)
    print(f"  {w:<15} perplexity={s['perplexity']:>8.2f}   avg_neg_logprob={s['avg_neg_logprob']:>5.2f}")

  mengabaikan     perplexity=    4.37   avg_neg_logprob= 1.47
  rumah           perplexity=    6.14   avg_neg_logprob= 1.82
  yang            perplexity=    6.35   avg_neg_logprob= 1.85
  ekstghak        perplexity=   42.26   avg_neg_logprob= 3.74
  petigh          perplexity=   88.84   avg_neg_logprob= 4.49
  tughun          perplexity=   26.31   avg_neg_logprob= 3.27
  xqz             perplexity=  224.57   avg_neg_logprob= 5.41


## 5. Calibrate the suspicion threshold

The threshold is **data-derived**: we compute perplexity for every word in
the training vocabulary, then set the threshold at the 95th percentile.
Words above this threshold during testing are flagged as suspicious.

This is principled because:
1. Training words are "real Indonesian" by construction
2. We expect 5% of training to have unusual patterns (rare valid words, loans)
3. Test words above this threshold are at least as unusual as the rarest 5%
   of known Indonesian words

In [34]:
training_perplexities = []
for word in vocab:
    s = score_word(word)
    training_perplexities.append(s["perplexity"])

training_perplexities = np.array(training_perplexities)

threshold_perplexity = float(np.percentile(training_perplexities, THRESHOLD_PERCENTILE))

print(f"Training perplexity stats:")
print(f"  min:    {training_perplexities.min():.2f}")
print(f"  median: {np.median(training_perplexities):.2f}")
print(f"  mean:   {training_perplexities.mean():.2f}")
print(f"  p75:    {np.percentile(training_perplexities, 75):.2f}")
print(f"  p95:    {np.percentile(training_perplexities, 95):.2f}")
print(f"  p99:    {np.percentile(training_perplexities, 99):.2f}")
print(f"  max:    {training_perplexities.max():.2f}")
print(f"\nThreshold (p{THRESHOLD_PERCENTILE}): {threshold_perplexity:.2f}")
print(f"  → Words with perplexity > {threshold_perplexity:.2f} will be flagged")

Training perplexity stats:
  min:    2.15
  median: 5.87
  mean:   6.72
  p75:    7.57
  p95:    12.18
  p99:    17.53
  max:    70.64

Threshold (p95): 12.18
  → Words with perplexity > 12.18 will be flagged


## 6. Word-level cache

Same trick as pyspellchecker v2: each unique token gets scored once, result
cached. The same word appears thousands of times in Parcor, but we only
compute its perplexity once.

In [35]:
_score_cache: dict[str, dict] = {}


def cached_score(word: str) -> dict:
    if word in _score_cache:
        return _score_cache[word]
    result = score_word(word)
    _score_cache[word] = result
    return result


# Pre-warm with very common words
for w in ["yang", "dan", "di", "ke", "dari", "dengan", "untuk", "ada", "ini",
          "itu", "tidak", "akan", "sudah", "atau", "juga", "saya", "dia",
          "kami", "kita", "mereka", "karena", "bisa"]:
    cached_score(w)
print(f"Cache primed with {len(_score_cache)} common words")

Cache primed with 22 common words


## 7. Lookup direction (which side is Indonesian)

In [36]:
lookup_df = pd.read_csv(LOOKUP_PATH)


def get_is_from_indonesia(input_path: str) -> tuple[bool, int]:
    filename = os.path.basename(input_path)
    try:
        file_id = filename.split("_")[0].strip()
    except IndexError:
        return False, -1
    for _, row in lookup_df.iterrows():
        raw_kamus = str(row["kamus"])
        match = re.search(r"\d+", raw_kamus)
        dict_id = match.group() if match else raw_kamus.strip()
        if dict_id == file_id:
            return True, int(row["is_from_indonesia"])
    return False, -1

## 8. Detect on a single sentence

In [37]:
TOKEN_RE = re.compile(r"[A-Za-z]{2,}")


def detect_sentence(sentence: str) -> dict:
    if not isinstance(sentence, str):
        return {"total_tokens": 0, "flagged": [], "perplexities": [], "mean_perplexity": 0.0}

    tokens = TOKEN_RE.findall(sentence)
    flagged = []
    perplexities = []
    all_perps = []

    for original in tokens:
        word = original.lower()
        s = cached_score(word)
        all_perps.append(s["perplexity"])
        if s["perplexity"] > threshold_perplexity:
            flagged.append(word)
            perplexities.append(f"{word}:{s['perplexity']:.1f}")

    mean_perp = float(np.mean(all_perps)) if all_perps else 0.0
    return {
        "total_tokens": len(tokens),
        "flagged": flagged,
        "perplexities": perplexities,
        "mean_perplexity": mean_perp,
    }

## 9. Process one Parcor file

In [38]:
def process_file(file_name: str) -> dict:
    input_file  = SRC_PARCOR_DIR / file_name
    output_file = DST_DIR / file_name.replace(".csv", "_ngram.csv")

    success, is_from_indonesia = get_is_from_indonesia(str(input_file))
    if not success:
        return {"file": file_name, "status": "skip_no_lookup"}
    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        return {"file": file_name, "status": f"read_error: {e}"}
    if "kalimat_asal" not in df.columns or "kalimat_tujuan" not in df.columns:
        return {"file": file_name, "status": "missing_columns"}

    indonesian_col = "kalimat_asal" if is_from_indonesia == 1 else "kalimat_tujuan"
    side_label = "asal" if is_from_indonesia == 1 else "tujuan"

    rows = []
    total_tokens, total_flagged = 0, 0
    perplexity_sums = []

    for _, row in df.iterrows():
        sentence = row[indonesian_col]
        det = detect_sentence(sentence)
        rows.append({
            "kalimat_asal":   row.get("kalimat_asal", ""),
            "kalimat_tujuan": row.get("kalimat_tujuan", ""),
            "indonesian_side": side_label,
            "total_tokens": det["total_tokens"],
            "flagged_tokens": ";".join(det["flagged"]),
            "n_flagged": len(det["flagged"]),
            "flag_rate": (
                len(det["flagged"]) / det["total_tokens"]
                if det["total_tokens"] > 0 else 0.0
            ),
            "perplexities": ";".join(det["perplexities"]),
            "mean_perplexity": round(det["mean_perplexity"], 2),
        })
        total_tokens += det["total_tokens"]
        total_flagged += len(det["flagged"])
        if det["mean_perplexity"] > 0:
            perplexity_sums.append(det["mean_perplexity"])

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_file, index=False)

    return {
        "file": file_name,
        "status": "ok",
        "rows": len(out_df),
        "total_tokens": total_tokens,
        "total_flagged": total_flagged,
        "flag_rate": total_flagged / total_tokens if total_tokens > 0 else 0.0,
        "mean_perplexity": float(np.mean(perplexity_sums)) if perplexity_sums else 0.0,
        "indonesian_side": side_label,
    }

## 10. Run the batch

In [39]:
files = sorted([
    f for f in os.listdir(SRC_PARCOR_DIR)
    if f.endswith(".csv") and not f.endswith("_audit.csv") and not f.startswith("_")
])
print(f"Found {len(files)} Parcor files to process\n")

start = time.time()
summaries = []
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_file, f): f for f in files}
    for count, future in enumerate(concurrent.futures.as_completed(futures), 1):
        result = future.result()
        summaries.append(result)
        elapsed = time.time() - start
        rate = count / elapsed if elapsed > 0 else 0
        eta = (len(files) - count) / rate if rate > 0 else 0
        if result["status"] == "ok":
            print(f"[{count}/{len(files)}] {result['file']:<30} "
                  f"{result['rows']:>5} rows  flag_rate={result['flag_rate']:.1%}  "
                  f"({elapsed:.1f}s elapsed, ETA {eta:.0f}s)")
        else:
            print(f"[{count}/{len(files)}] {result['file']:<30} STATUS: {result['status']}")

elapsed_total = time.time() - start
print(f"\nDone in {elapsed_total:.1f}s ({elapsed_total / max(1, len(files)):.1f}s per file avg)")
print(f"Cache final: {len(_score_cache):,} unique words scored")

Found 68 Parcor files to process

[1/68] 11_Parcor.csv                    609 rows  flag_rate=18.3%  (0.7s elapsed, ETA 48s)
[2/68] 13_Parcor.csv                    588 rows  flag_rate=15.8%  (0.9s elapsed, ETA 29s)
[3/68] 10_Parcor.csv                   2942 rows  flag_rate=9.7%  (1.0s elapsed, ETA 22s)
[4/68] 12_Parcor.csv                   4810 rows  flag_rate=20.6%  (1.5s elapsed, ETA 25s)
[5/68] 15_Parcor.csv                    711 rows  flag_rate=15.0%  (1.5s elapsed, ETA 19s)
[6/68] 14_Parcor.csv                    997 rows  flag_rate=5.0%  (1.7s elapsed, ETA 18s)
[7/68] 16_Parcor.csv                   1889 rows  flag_rate=7.4%  (2.0s elapsed, ETA 17s)
[8/68] 17_Parcor.csv                   1673 rows  flag_rate=5.2%  (2.0s elapsed, ETA 15s)
[9/68] 18_Parcor.csv                   2341 rows  flag_rate=6.9%  (2.0s elapsed, ETA 13s)
[10/68] 20_Parcor.csv                   1719 rows  flag_rate=1.0%  (3.0s elapsed, ETA 18s)
[11/68] 1_Parcor.csv                    2914 rows  flag_rate=

## 11. Per-dict summary

In [40]:
ok_summaries = [s for s in summaries if s["status"] == "ok"]
summary_df = pd.DataFrame(ok_summaries)
summary_df["dict_id"] = summary_df["file"].apply(
    lambda f: re.match(r"^(\d+)_", f).group(1) if re.match(r"^(\d+)_", f) else None
)
summary_df = summary_df.sort_values(
    "dict_id", key=lambda s: s.astype(int)
).reset_index(drop=True)

summary_df.to_csv(DST_DIR / "_per_dict_summary.csv", index=False)

print(f"=== Per-dict summary (Strategy: {strategy_tag}, n-gram method) ===")
print(summary_df[["dict_id", "rows", "total_tokens", "total_flagged",
                  "flag_rate", "mean_perplexity"]].head(20).to_string(index=False))
print(f"\nTotal files OK: {len(summary_df)}")
print(f"Total tokens checked: {summary_df['total_tokens'].sum():,}")
print(f"Total flagged: {summary_df['total_flagged'].sum():,}")
print(f"Mean flag rate: {summary_df['flag_rate'].mean():.1%}")
print(f"Mean perplexity (across files): {summary_df['mean_perplexity'].mean():.2f}")

=== Per-dict summary (Strategy: strategy_C, n-gram method) ===
dict_id  rows  total_tokens  total_flagged  flag_rate  mean_perplexity
      1  2914         22573           2239   0.099189         7.858003
      2  2047         19699           1737   0.088177        14.944479
      3   125         34506           7247   0.210021        15.057752
      4  2658         17052            951   0.055771         7.488923
      5   697          1803            434   0.240710        12.396778
      8   413          5920           1015   0.171453        12.629017
      9  1049         10158            963   0.094802         9.546795
     10  2942         16472           1599   0.097074         8.847936
     11   609         29231           5350   0.183025        19.844073
     12  4810         29933           6176   0.206327        13.310697
     13   588         32387           5107   0.157687         9.277531
     14   997         47216           2379   0.050385         7.561136
     15   711 

## 12. Top suspicious tokens

Aggregate flagged tokens across all files. Output: ranked list of unique
tokens with how many times they appeared and their perplexity score.

This is the input to the pattern miner notebook.

In [41]:
all_flagged = Counter()
flagged_perplexity = {}

for f in DST_DIR.glob("*_Parcor_ngram.csv"):
    try:
        df = pd.read_csv(f)
    except Exception:
        continue
    for cell in df["flagged_tokens"].dropna():
        if isinstance(cell, str) and cell:
            for tok in cell.split(";"):
                if tok:
                    all_flagged[tok] += 1
    for cell in df["perplexities"].dropna():
        if isinstance(cell, str) and cell:
            for pair in cell.split(";"):
                if ":" in pair:
                    tok, perp = pair.rsplit(":", 1)
                    try:
                        flagged_perplexity[tok] = float(perp)
                    except ValueError:
                        pass

top_df = pd.DataFrame([
    {"token": tok, "occurrences": cnt, "perplexity": flagged_perplexity.get(tok, float("nan"))}
    for tok, cnt in all_flagged.most_common(200)
])
top_df.to_csv(DST_DIR / "_top_suspicious_tokens.csv", index=False)

print(f"Total unique flagged tokens: {len(all_flagged):,}")
print(f"\n=== Top 30 most-flagged tokens (across the corpus) ===")
print(top_df.head(30).to_string(index=False))

Total unique flagged tokens: 68,410

=== Top 30 most-flagged tokens (across the corpus) ===
   token  occurrences  perplexity
     utk         1202        16.7
     anu         1002        13.2
     meu          696        39.5
      vt          627        12.7
     org          530        21.3
   jeung          522        19.8
      km          509        15.3
     rsd          486       389.1
    jite          430        14.5
      sj          393        20.5
     iye          377        18.5
      ml          341        12.7
   kiota          337        14.4
     wau          321        13.1
  ltuyuo          316        49.4
    tiyo          315        31.6
    wonu          290        21.2
     etu          290        12.6
      wo          269        13.0
     ito          265        13.1
      kr          247        12.6
 pasteyi          242        55.3
    geus          227        37.6
     ate          220        16.5
nglengek          216        14.1
     hna          213   

## 13. Aggregate stats

In [42]:
aggregate = {
    "method": "ngram",
    "strategy": strategy_tag,
    "dict_path": DICT_PATH,
    "vocab_size": len(vocab),
    "ngram_order": NGRAM_ORDER,
    "smoothing_k": SMOOTHING_K,
    "threshold_percentile": THRESHOLD_PERCENTILE,
    "threshold_perplexity": round(threshold_perplexity, 2),
    "files_processed": len(summary_df),
    "total_tokens_checked": int(summary_df["total_tokens"].sum()),
    "total_flagged_tokens": int(summary_df["total_flagged"].sum()),
    "unique_flagged_tokens": len(all_flagged),
    "mean_flag_rate":   round(float(summary_df["flag_rate"].mean()), 4),
    "median_flag_rate": round(float(summary_df["flag_rate"].median()), 4),
    "p95_flag_rate":    round(float(summary_df["flag_rate"].quantile(0.95)), 4),
    "elapsed_seconds": round(elapsed_total, 1),
}

agg_df = pd.DataFrame([aggregate])
agg_df.to_csv(DST_DIR / "_aggregate.csv", index=False)
for k, v in aggregate.items():
    print(f"  {k:<25} {v}")
print(f"\nAll outputs written to: {DST_DIR.resolve()}")

  method                    ngram
  strategy                  strategy_C
  dict_path                 ../spellCheckDicts/indonesian_dict_strategy_C.json
  vocab_size                22601
  ngram_order               3
  smoothing_k               0.01
  threshold_percentile      95
  threshold_perplexity      12.18
  files_processed           68
  total_tokens_checked      1434567
  total_flagged_tokens      151258
  unique_flagged_tokens     68410
  mean_flag_rate            0.1044
  median_flag_rate          0.0885
  p95_flag_rate             0.2184
  elapsed_seconds           12.3

All outputs written to: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\12. Parallel Corpus - Spellcheck Ngram\strategy_C


## 14. Comparing with pyspellchecker

If you've already run `PostProcessing pyspellchecker v2.ipynb` with the same
strategy, you can do a quick four-way analysis:

In [43]:
PYSPELL_DIR = Path(f"../Ekstraksi/12. Parallel Corpus - Spellcheck Detection/{strategy_tag}")

if PYSPELL_DIR.exists():
    pyspell_top = pd.read_csv(PYSPELL_DIR / "_top_unknown_tokens.csv")
    pyspell_set = set(pyspell_top["token"])
    ngram_set = set(top_df["token"])

    both = pyspell_set & ngram_set
    only_pyspell = pyspell_set - ngram_set
    only_ngram = ngram_set - pyspell_set

    print(f"Caught by both methods:        {len(both):,}")
    print(f"Caught by pyspell only:        {len(only_pyspell):,}")
    print(f"Caught by n-gram only:         {len(only_ngram):,}")
    print()
    print(f"=== 10 examples — caught by BOTH (high-confidence errors) ===")
    print(list(both)[:10])
    print(f"\n=== 10 examples — caught by PYSPELL only (unknown but well-formed) ===")
    print(list(only_pyspell)[:10])
    print(f"\n=== 10 examples — caught by NGRAM only (in dict but oddly-shaped) ===")
    print(list(only_ngram)[:10])
else:
    print(f"Pyspellchecker output not found at {PYSPELL_DIR}")
    print("Run PostProcessing pyspellchecker v2.ipynb with the same strategy first.")

Caught by both methods:        50
Caught by pyspell only:        50
Caught by n-gram only:         150

=== 10 examples — caught by BOTH (high-confidence errors) ===
['tou', 'erbage', 'lhut', 'nglengek', 'ltuyuo', 'wagu', 'bh', 'itoh', 'jite', 'ente']

=== 10 examples — caught by PYSPELL only (unknown but well-formed) ===
['kerina', 'nonggomai', 'onggo', 'amaitu', 'idek', 'gumai', 'kalawan', 'takinia', 'jalma', 'keu']

=== 10 examples — caught by NGRAM only (in dict but oddly-shaped) ===
['ml', 'uu', 'gg', 'prm', 'bielo', 'ij', 'sallgkut', 'leuwih', 'ibonia', 'ano']
